# 06 · Interactive Visual Analytics with Folium

**IBM Applied Data Science Capstone · SpaceX Falcon 9 first-stage landing prediction**  
Author: Martim Ramos

Mark all launch sites, show successful/failed launches per site with marker clusters, and calculate the distance from a launch site to its proximities. Input: `data/spacex_launch_geo.csv`.

In [1]:
import folium
import pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians
spacex_df = pd.read_csv('data/spacex_launch_geo.csv')
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610746


## Task 1: Mark all launch sites on a map

In [2]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
for _, site in launch_sites_df.iterrows():
    coordinate = [site['Lat'], site['Long']]
    folium.Circle(coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup(site['Launch Site'])).add_to(site_map)
    folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % site['Launch Site'])).add_to(site_map)
site_map

All launch sites are close to the coast; the Florida sites are at ~28.5°N, close to the equator (by US standards), and VAFB is on the California coast.

## Task 2: Mark the success/failed launches for each site on the map

In [3]:
marker_cluster = MarkerCluster()
spacex_df['marker_color'] = spacex_df['class'].apply(lambda c: 'green' if c == 1 else 'red')
site_map.add_child(marker_cluster)
for _, record in spacex_df.iterrows():
    marker = folium.Marker(location=[record['Lat'], record['Long']],
                           icon=folium.Icon(color='white', icon_color=record['marker_color']))
    marker_cluster.add_child(marker)
site_map

In [4]:
spacex_df.groupby('Launch Site')['class'].agg(['sum', 'count', 'mean'])

,sum,count,mean
Launch Site,,,
CCAFS LC-40,7,26,0.269231
CCAFS SLC-40,3,7,0.428571
KSC LC-39A,10,13,0.769231
VAFB SLC-4E,4,10,0.400000


KSC LC-39A has the highest success ratio (10 of 13 = 76.9%).

## Task 3: Calculate the distances between a launch site and its proximities

In [5]:
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(position='topright', separator=' Long: ', empty_string='NaN', lng_first=False,
                               num_digits=20, prefix='Lat:', lat_formatter=formatter, lng_formatter=formatter)
site_map.add_child(mouse_position)

def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

In [6]:
launch_site_lat, launch_site_lon = 28.563197177407144, -80.57682003124195   # CCAFS SLC-40
# nearest points read with MousePosition (checked against Natural Earth 10 m layers)
proximities = {
    'Coastline':        (28.565810657550912, -80.57272904173466),
    'Highway (SR-405)': (28.525302, -80.664366),
    'Railway':          (28.476666, -80.785),
    'City (Titusville)': (28.612348, -80.807791),
}
colors = {'Coastline': 'blue', 'Highway (SR-405)': 'orange', 'Railway': 'black', 'City (Titusville)': 'purple'}
for name, (lat, lon) in proximities.items():
    d = calculate_distance(launch_site_lat, launch_site_lon, lat, lon)
    print(f'{name}: {d:.2f} km')
    folium.Marker([lat, lon], icon=DivIcon(icon_size=(200, 36), icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:%s;"><b>%s: %.2f KM</b></div>' % (colors[name], name, d))).add_to(site_map)
    folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], [lat, lon]], weight=2, color=colors[name]).add_to(site_map)
site_map.location = [launch_site_lat, launch_site_lon]
site_map

Coastline: 0.49 km
Highway (SR-405): 9.53 km
Railway: 22.50 km
City (Titusville): 23.20 km


**Findings:** the launch site is very close to the coastline (~0.5 km) for safe over-water flight, but kept well away from cities, highways and mainline railways for public safety.